# Phase 2 Factorial Analysis

Statistical analysis for the 48-condition factorial experiment testing:
- **H1**: Modality (image-only vs text+image)
- **H5**: Example ordering (canonical-first, canonical-last, random)
- **H7**: Hard negatives (without, with)
- **H9**: Temperature (0.0, 0.3, 0.7, 1.0)

**Design**: 2 × 3 × 2 × 4 = 48 conditions, each with 5 passes on 20 holdout tiles.

---

## Setup

In [ ]:
import json
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

# Statistical packages
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.multitest import multipletests
from scipy import stats

# Project imports
import sys
sys.path.insert(0, str(Path.cwd().parent))
from scripts.lib_advanced_metrics import calculate_f1_internal

# Configuration
PROJECT_ROOT = Path.cwd().parent
RESULTS_DIR = PROJECT_ROOT / 'outputs' / 'phase2-factorial'
ALPHA = 0.05
FDR_Q = 0.05

print(f"Results directory: {RESULTS_DIR}")
print(f"Exists: {RESULTS_DIR.exists()}")

## 1. Load Results

Load all condition results from the study manifest and compute F1 scores.

In [ ]:
def load_study_results(results_dir: Path) -> pd.DataFrame:
    """
    Load all condition results and compute metrics.
    
    Returns DataFrame with columns:
    - condition_id
    - modality, ordering, hard_negatives, temperature (factor levels)
    - f1, precision, recall (metrics)
    """
    manifest_path = results_dir / 'study_manifest.json'
    if not manifest_path.exists():
        raise FileNotFoundError(f"Study manifest not found: {manifest_path}")
    
    with open(manifest_path) as f:
        manifest = json.load(f)
    
    rows = []
    for condition in manifest['conditions']:
        condition_id = condition['condition_id']
        
        # Load metrics for this condition
        metrics_file = results_dir / 'results' / f'{condition_id}_metrics.json'
        if not metrics_file.exists():
            print(f"Warning: Metrics not found for {condition_id}")
            continue
        
        with open(metrics_file) as f:
            metrics = json.load(f)
        
        row = {
            'condition_id': condition_id,
            'modality': condition['factor_levels']['modality'],
            'ordering': condition['factor_levels']['ordering'],
            'hard_negatives': condition['factor_levels']['hard_negatives'],
            'temperature': condition['temperature'],
            'f1': metrics.get('f1', np.nan),
            'precision': metrics.get('precision', np.nan),
            'recall': metrics.get('recall', np.nan),
            'tp': metrics.get('tp', 0),
            'fp': metrics.get('fp', 0),
            'fn': metrics.get('fn', 0),
        }
        rows.append(row)
    
    df = pd.DataFrame(rows)
    print(f"Loaded {len(df)} conditions")
    return df

# Load data (uncomment when results exist)
# df = load_study_results(RESULTS_DIR)
# df.head()

## 2. Descriptive Statistics

In [ ]:
# Summary statistics by factor
# Uncomment when data is loaded

# print("=== F1 by Modality ===")
# print(df.groupby('modality')['f1'].describe())

# print("\n=== F1 by Ordering ===")
# print(df.groupby('ordering')['f1'].describe())

# print("\n=== F1 by Hard Negatives ===")
# print(df.groupby('hard_negatives')['f1'].describe())

# print("\n=== F1 by Temperature ===")
# print(df.groupby('temperature')['f1'].describe())

## 3. Four-Way ANOVA

Test main effects and two-way interactions.

In [ ]:
def run_factorial_anova(df: pd.DataFrame) -> pd.DataFrame:
    """
    Run 4-way ANOVA with all main effects and 2-way interactions.
    
    Model: F1 ~ M + O + H + T + M:O + M:H + M:T + O:H + O:T + H:T
    """
    # Convert temperature to categorical for ANOVA
    df = df.copy()
    df['temperature_cat'] = df['temperature'].astype(str)
    
    # Fit model with main effects and 2-way interactions
    formula = (
        'f1 ~ C(modality) + C(ordering) + C(hard_negatives) + C(temperature_cat) + '
        'C(modality):C(ordering) + C(modality):C(hard_negatives) + C(modality):C(temperature_cat) + '
        'C(ordering):C(hard_negatives) + C(ordering):C(temperature_cat) + '
        'C(hard_negatives):C(temperature_cat)'
    )
    
    model = ols(formula, data=df).fit()
    anova_table = sm.stats.anova_lm(model, typ=2)
    
    return anova_table

# Run ANOVA (uncomment when data is loaded)
# anova_results = run_factorial_anova(df)
# print(anova_results)

## 4. FDR Correction

Apply Benjamini-Hochberg FDR correction at q = 0.05.

In [ ]:
def apply_fdr_correction(anova_table: pd.DataFrame, q: float = 0.05) -> pd.DataFrame:
    """
    Apply Benjamini-Hochberg FDR correction to ANOVA p-values.
    
    Args:
        anova_table: ANOVA results with 'PR(>F)' column
        q: FDR threshold (default 0.05)
    
    Returns:
        ANOVA table with added 'p_fdr' and 'significant_fdr' columns
    """
    result = anova_table.copy()
    
    # Get p-values (excluding Residual row)
    mask = result.index != 'Residual'
    p_values = result.loc[mask, 'PR(>F)'].values
    
    # Apply BH correction
    rejected, p_corrected, _, _ = multipletests(p_values, alpha=q, method='fdr_bh')
    
    # Add to table
    result.loc[mask, 'p_fdr'] = p_corrected
    result.loc[mask, 'significant_fdr'] = rejected
    
    return result

# Apply FDR (uncomment when data is loaded)
# anova_fdr = apply_fdr_correction(anova_results, q=FDR_Q)
# print(anova_fdr[['F', 'PR(>F)', 'p_fdr', 'significant_fdr']])

## 5. Effect Sizes

Calculate effect sizes (partial eta-squared, Cohen's d) for significant effects.

In [ ]:
def calculate_effect_sizes(df: pd.DataFrame, anova_table: pd.DataFrame) -> pd.DataFrame:
    """
    Calculate partial eta-squared for each effect.
    
    Partial η² = SS_effect / (SS_effect + SS_residual)
    """
    result = anova_table.copy()
    ss_residual = result.loc['Residual', 'sum_sq']
    
    partial_eta_sq = []
    for idx in result.index:
        if idx == 'Residual':
            partial_eta_sq.append(np.nan)
        else:
            ss_effect = result.loc[idx, 'sum_sq']
            eta = ss_effect / (ss_effect + ss_residual)
            partial_eta_sq.append(eta)
    
    result['partial_eta_sq'] = partial_eta_sq
    
    # Interpretation thresholds (Cohen, 1988)
    # Small: 0.01, Medium: 0.06, Large: 0.14
    def interpret_eta(eta):
        if pd.isna(eta):
            return ''
        if eta < 0.01:
            return 'negligible'
        elif eta < 0.06:
            return 'small'
        elif eta < 0.14:
            return 'medium'
        else:
            return 'large'
    
    result['effect_size'] = result['partial_eta_sq'].apply(interpret_eta)
    
    return result

# Calculate effect sizes (uncomment when data is loaded)
# anova_effects = calculate_effect_sizes(df, anova_fdr)
# print(anova_effects[['F', 'PR(>F)', 'p_fdr', 'significant_fdr', 'partial_eta_sq', 'effect_size']])

## 6. Hypothesis-Specific Tests

In [ ]:
def test_h1_modality(df: pd.DataFrame) -> dict:
    """
    H1: Text modality has no significant effect.
    Two-tailed test for equivalence.
    """
    image_only = df[df['modality'] == 'image-only']['f1']
    text_image = df[df['modality'] == 'text-image']['f1']
    
    # Two-tailed t-test
    t_stat, p_value = stats.ttest_ind(image_only, text_image)
    
    # Effect size (Cohen's d)
    pooled_std = np.sqrt((image_only.std()**2 + text_image.std()**2) / 2)
    cohens_d = (image_only.mean() - text_image.mean()) / pooled_std
    
    # 95% CI for difference
    diff = image_only.mean() - text_image.mean()
    se = np.sqrt(image_only.var()/len(image_only) + text_image.var()/len(text_image))
    ci_lower = diff - 1.96 * se
    ci_upper = diff + 1.96 * se
    
    return {
        'hypothesis': 'H1',
        'test': 'Two-tailed t-test',
        'image_only_mean': image_only.mean(),
        'text_image_mean': text_image.mean(),
        'difference': diff,
        'ci_95': (ci_lower, ci_upper),
        't_statistic': t_stat,
        'p_value': p_value,
        'cohens_d': cohens_d,
        'significant': p_value < ALPHA,
    }

# Test H1 (uncomment when data is loaded)
# h1_result = test_h1_modality(df)
# print(json.dumps(h1_result, indent=2, default=str))

In [ ]:
def test_h9_temperature(df: pd.DataFrame) -> dict:
    """
    H9: Temperature = 1.0 performs at least as well as lower temperatures.
    One-way ANOVA across temperature conditions.
    """
    # Group by temperature
    groups = [group['f1'].values for name, group in df.groupby('temperature')]
    
    # One-way ANOVA
    f_stat, p_value = stats.f_oneway(*groups)
    
    # Means by temperature
    means = df.groupby('temperature')['f1'].mean().to_dict()
    
    # Pairwise comparisons with T=1.0
    t1_data = df[df['temperature'] == 1.0]['f1']
    comparisons = {}
    for temp in [0.0, 0.3, 0.7]:
        other_data = df[df['temperature'] == temp]['f1']
        t, p = stats.ttest_ind(t1_data, other_data)
        comparisons[f'T1.0_vs_T{temp}'] = {'t': t, 'p': p}
    
    return {
        'hypothesis': 'H9',
        'test': 'One-way ANOVA',
        'means_by_temp': means,
        'f_statistic': f_stat,
        'p_value': p_value,
        'significant': p_value < ALPHA,
        'pairwise_vs_T1': comparisons,
    }

# Test H9 (uncomment when data is loaded)
# h9_result = test_h9_temperature(df)
# print(json.dumps(h9_result, indent=2, default=str))

## 7. Visualisations

In [ ]:
def plot_factorial_results(df: pd.DataFrame):
    """
    Generate visualisations for factorial results.
    """
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    
    # 1. F1 by Modality
    ax = axes[0, 0]
    sns.boxplot(data=df, x='modality', y='f1', ax=ax)
    ax.set_title('H1: F1 by Modality')
    ax.set_xlabel('Modality')
    ax.set_ylabel('F1 Score')
    
    # 2. F1 by Ordering
    ax = axes[0, 1]
    sns.boxplot(data=df, x='ordering', y='f1', ax=ax)
    ax.set_title('H5: F1 by Example Ordering')
    ax.set_xlabel('Ordering')
    ax.set_ylabel('F1 Score')
    
    # 3. F1 by Hard Negatives
    ax = axes[1, 0]
    sns.boxplot(data=df, x='hard_negatives', y='f1', ax=ax)
    ax.set_title('H7: F1 by Hard Negatives')
    ax.set_xlabel('Hard Negatives')
    ax.set_ylabel('F1 Score')
    
    # 4. F1 by Temperature
    ax = axes[1, 1]
    sns.boxplot(data=df, x='temperature', y='f1', ax=ax)
    ax.set_title('H9: F1 by Temperature')
    ax.set_xlabel('Temperature')
    ax.set_ylabel('F1 Score')
    
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'factorial_main_effects.png', dpi=150)
    plt.show()

# Generate plots (uncomment when data is loaded)
# plot_factorial_results(df)

In [ ]:
def plot_interaction(df: pd.DataFrame, factor1: str, factor2: str):
    """
    Plot interaction between two factors.
    """
    plt.figure(figsize=(8, 6))
    
    # Compute means
    means = df.groupby([factor1, factor2])['f1'].mean().unstack()
    
    # Plot
    means.plot(marker='o')
    plt.title(f'Interaction: {factor1} × {factor2}')
    plt.xlabel(factor1)
    plt.ylabel('F1 Score')
    plt.legend(title=factor2)
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / f'interaction_{factor1}_{factor2}.png', dpi=150)
    plt.show()

# Plot interactions (uncomment when data is loaded)
# plot_interaction(df, 'modality', 'hard_negatives')
# plot_interaction(df, 'ordering', 'temperature')

## 8. Summary Table

In [ ]:
def generate_summary_table(df: pd.DataFrame, anova_results: pd.DataFrame) -> pd.DataFrame:
    """
    Generate publication-ready summary table.
    """
    # Extract significant effects
    sig_effects = anova_results[anova_results['significant_fdr'] == True].index.tolist()
    
    summary = []
    for factor in ['modality', 'ordering', 'hard_negatives', 'temperature']:
        for level in df[factor].unique():
            subset = df[df[factor] == level]
            summary.append({
                'Factor': factor,
                'Level': level,
                'N': len(subset),
                'F1_mean': subset['f1'].mean(),
                'F1_std': subset['f1'].std(),
                'Precision_mean': subset['precision'].mean(),
                'Recall_mean': subset['recall'].mean(),
            })
    
    return pd.DataFrame(summary)

# Generate summary (uncomment when data is loaded)
# summary_df = generate_summary_table(df, anova_effects)
# summary_df.to_csv(RESULTS_DIR / 'factorial_summary.csv', index=False)
# summary_df

## 9. Export Results

In [ ]:
def export_all_results(df, anova_results, hypothesis_tests):
    """
    Export all analysis results to files.
    """
    # 1. Raw data
    df.to_csv(RESULTS_DIR / 'condition_metrics.csv', index=False)
    
    # 2. ANOVA table
    anova_results.to_csv(RESULTS_DIR / 'anova_results.csv')
    
    # 3. Hypothesis tests
    with open(RESULTS_DIR / 'hypothesis_tests.json', 'w') as f:
        json.dump(hypothesis_tests, f, indent=2, default=str)
    
    print(f"Results exported to {RESULTS_DIR}")

# Export (uncomment when data is loaded)
# export_all_results(df, anova_effects, {'H1': h1_result, 'H9': h9_result})

---

## Notes

**Statistical Methods**:
- Four-way factorial ANOVA with Type II sum of squares
- Benjamini-Hochberg FDR correction at q = 0.05
- Effect sizes: partial eta-squared (η²ₚ)
- Pairwise comparisons: Tukey HSD or independent t-tests

**Interpretation Thresholds** (Cohen, 1988):
- η²ₚ < 0.01: negligible
- η²ₚ = 0.01-0.06: small
- η²ₚ = 0.06-0.14: medium
- η²ₚ > 0.14: large

**Required Packages**:
```bash
pip install statsmodels scipy pandas numpy matplotlib seaborn
```